In [2]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import openml
import os
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer, OrdinalEncoder
from src.data_loader import load_dataset_info


load_dotenv()
openml.config.apikey = os.getenv('OPENML_API_KEY')
openml.config.cache_directory = os.path.expanduser('~/.openml/cache')

# Loading  the enriched summary so we have metadata for all 45 datasets
df = pd.read_csv('../results/dataset_summary_enriched.csv')

print(f'Loaded {len(df)} datasets')

Loaded 55 datasets


In [3]:
#first understanding about the dataset 


task    = openml.tasks.get_task(361055)
dataset = task.get_dataset()

X, y, categorical_indicator, attribute_names = dataset.get_data(
    target=task.target_name
)

X = pd.DataFrame(X, columns=attribute_names)
y = pd.Series(y)

# Understanding  what we're working with before touching anything
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"y dtype: {y.dtype}")
print(f"Categorical features: {sum(categorical_indicator)}")
print(f"Numerical features: {sum(1 for c in categorical_indicator if not c)}")
print(f"\nFirst few rows of X:")
print(X.head())
print(f"\nFirst few values of y:")
print(y.head())

X shape: (16714, 10)
y shape: (16714,)
y dtype: category
Categorical features: 0
Numerical features: 10

First few rows of X:
   RevolvingUtilizationOfUnsecuredLines  age  \
0                              0.006999   38   
1                              0.704592   63   
2                              0.063113   57   
3                              0.368397   68   
4                              1.000000   34   

   NumberOfTime30-59DaysPastDueNotWorse  DebtRatio  MonthlyIncome  \
0                                     0   0.302150         5440.0   
1                                     0   0.471441         8000.0   
2                                     0   0.068586         5000.0   
3                                     0   0.296273         6250.0   
4                                     1   0.000000         3500.0   

   NumberOfOpenCreditLinesAndLoans  NumberOfTimes90DaysLate  \
0                                4                        0   
1                                9          

In [4]:
#Sample Capping 

def cap_samples(X, y, max_samples=10000, random_state=42):
    if len(X) > max_samples:
        # We use pandas sample() here because it works naturally with
        # DataFrames and handles the index alignment between X and y automatically
        X_sampled = X.sample(n=max_samples, random_state=random_state)
        y_sampled = y.loc[X_sampled.index]
        return X_sampled.reset_index(drop=True), y_sampled.reset_index(drop=True)
    return X.reset_index(drop=True), y.reset_index(drop=True)

# Testing  it on our loaded dataset
X_capped, y_capped = cap_samples(X, y)
print(f"Before capping: {X.shape}")
print(f"After capping:  {X_capped.shape}")
print(f"y shape after capping: {y_capped.shape}")

Before capping: (16714, 10)
After capping:  (10000, 10)
y shape after capping: (10000,)


In [ ]:
#Train-test split Function 

def split_data(X, y, test_size=0.2, random_state=42):
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state
    )
    
    X_train = X_train.reset_index(drop=True)
    X_test  = X_test.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)
    y_test  = y_test.reset_index(drop=True)
    
    return X_train, X_test, y_train, y_test

# Testing it on our capped dataset
X_train, X_test, y_train, y_test = split_data(X_capped, y_capped)

print(f"Training set:  {X_train.shape}")
print(f"Test set:      {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")

Training set:  (8000, 10)
Test set:      (2000, 10)
y_train shape: (8000,)
y_test shape:  (2000,)


In [ ]:
#scaling numerical features using QuantileTransformer
def scale_numerical_features(X_train, X_test, categorical_indicator, attribute_names):
    
    # Identifing which columns are numerical
    num_cols = [col for col, is_cat in zip(attribute_names, categorical_indicator)
                if not is_cat]
    
    if len(num_cols) == 0:
        return X_train, X_test
    
    X_train_scaled = X_train.copy()
    X_test_scaled  = X_test.copy()
    
    # Fit the transformer on training data only
    qt = QuantileTransformer(output_distribution='normal', random_state=42)
    
    # Fit and transform training set, then transform test set using
    # the same fitted transformer — this is the critical step
    X_train_scaled[num_cols] = qt.fit_transform(X_train[num_cols])
    X_test_scaled[num_cols]  = qt.transform(X_test[num_cols])
    
    return X_train_scaled, X_test_scaled

# Test on our split dataset
# For this numerical dataset, categorical_indicator is all False
X_train_scaled, X_test_scaled = scale_numerical_features(
    X_train, X_test, categorical_indicator, attribute_names
)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"\nBefore scaling — MonthlyIncome stats:")
print(X_train['MonthlyIncome'].describe().round(2))
print(f"\nAfter scaling — MonthlyIncome stats:")
print(X_train_scaled['MonthlyIncome'].describe().round(2))

X_train_scaled shape: (8000, 10)

Before scaling — MonthlyIncome stats:
count     8000.00
mean      6021.68
std       5007.65
min          0.00
25%       3120.00
50%       5000.00
75%       7500.00
max      97000.00
Name: MonthlyIncome, dtype: float64

After scaling — MonthlyIncome stats:
count    8000.00
mean       -0.03
std         1.10
min        -5.20
25%        -0.67
50%         0.02
75%         0.67
max         5.20
Name: MonthlyIncome, dtype: float64


In [ ]:
#Enoding categorical features using OrdinalEncoder

def encode_categorical_features(X_train, X_test, categorical_indicator, attribute_names):
    
    cat_cols = [col for col, is_cat in zip(attribute_names, categorical_indicator)
                if is_cat]
    
    if len(cat_cols) == 0:
        return X_train, X_test
    
    X_train_encoded = X_train.copy()
    X_test_encoded  = X_test.copy()
    
    # Fitting  on the training data only
    encoder = OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1
    )
    
    X_train_encoded[cat_cols] = encoder.fit_transform(X_train[cat_cols])
    X_test_encoded[cat_cols]  = encoder.transform(X_test[cat_cols])
    
    return X_train_encoded, X_test_encoded


X_train_enc, X_test_enc = encode_categorical_features(
    X_train_scaled, X_test_scaled,
    categorical_indicator, attribute_names
)


X_train_enc shape: (8000, 10)


In [10]:
#Transforming the target variable

def transform_target(y_train, y_test, task_type):
    
    if task_type == 'regression':
        y_train_vals = y_train.astype(float).values.reshape(-1, 1)
        y_test_vals  = y_test.astype(float).values.reshape(-1, 1)
        
        qt = QuantileTransformer(output_distribution='normal', random_state=42)
        
        # Fitting  on training target only — never on the full y
        y_train_transformed = qt.fit_transform(y_train_vals).ravel()
        y_test_transformed  = qt.transform(y_test_vals).ravel()
        
        return y_train_transformed, y_test_transformed, qt
    
    else:
        le = LabelEncoder()
        y_train_transformed = le.fit_transform(y_train.astype(str))
        y_test_transformed  = le.transform(y_test.astype(str))
        
        return y_train_transformed, y_test_transformed, le

# Testing  on our current classification dataset
from sklearn.preprocessing import LabelEncoder

y_train_transformed, y_test_transformed, transformer = transform_target(
    y_train, y_test, task_type='classification'
)


In [13]:
#Wrapping everything in a single function

def preprocess_dataset(task_id, suite_name):
    """
    Full preprocessing pipeline for a single dataset.
    Takes a task_id and suite_name, loads the raw data from OpenML,
    and returns model-ready train and test sets with all transformations
    applied in the correct order:

        1. Cap samples at 10,000
        2. Train/test split 80/20
        3. Scale numerical features with QuantileTransformer
        4. Encode categorical features with OrdinalEncoder
        5. Transform target variable

    Returns X_train, X_test, y_train, y_test, and the target transformer
    so predictions can be inverse-transformed back to the original scale
    for regression tasks.
    """
    # Step 1 — Loading the  raw data from OpenML
    task    = openml.tasks.get_task(task_id)
    dataset = task.get_dataset()
    X, y, categorical_indicator, attribute_names = dataset.get_data(
        target=task.target_name
    )
    X = pd.DataFrame(X, columns=attribute_names)
    y = pd.Series(y)
    
    # Step 2 — Capping the  samples at 10,000
    X, y = cap_samples(X, y, max_samples=10000, random_state=42)
    
    # Step 3 — Split into train and test
    X_train, X_test, y_train, y_test = split_data(X, y)
    
    # Step 4 — Scaling the  numerical features
    X_train, X_test = scale_numerical_features(
        X_train, X_test, categorical_indicator, attribute_names
    )
    
    # Step 5 — Encoding the  categorical features
    X_train, X_test = encode_categorical_features(
        X_train, X_test, categorical_indicator, attribute_names
    )
    
    # Step 6 — Transforming the  target variable
    task_type = 'regression' if 'regression' in suite_name else 'classification'
    y_train, y_test, target_transformer = transform_target(
        y_train, y_test, task_type
    )
    
    return X_train, X_test, y_train, y_test, target_transformer




In [14]:
# Testing  the full pipeline on our numerical classification dataset

X_train, X_test, y_train, y_test, transformer = preprocess_dataset(
    task_id=361055,
    suite_name='numerical_classification'
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")
print(f"y_train unique values: {np.unique(y_train)}")
print(f"y_train dtype: {y_train.dtype}")
print(f"\nX_train sample:")
print(X_train.head())

X_train shape: (8000, 10)
X_test shape:  (2000, 10)
y_train shape: (8000,)
y_test shape:  (2000,)
y_train unique values: [0 1]
y_train dtype: int64

X_train sample:
   RevolvingUtilizationOfUnsecuredLines       age  \
0                             -5.199338  0.064027   
1                              0.546864 -2.085356   
2                             -1.297789 -1.756055   
3                             -0.177444 -0.565738   
4                             -5.199338 -2.542349   

   NumberOfTime30-59DaysPastDueNotWorse  DebtRatio  MonthlyIncome  \
0                              0.731515   0.116234       0.092972   
1                             -5.199338   0.023819      -1.537691   
2                             -5.199338  -1.134723      -0.241963   
3                             -5.199338   0.084771      -0.060256   
4                             -5.199338  -5.199338      -1.866795   

   NumberOfOpenCreditLinesAndLoans  NumberOfTimes90DaysLate  \
0                        -0.592444    

In [15]:
# Testing  the full pipeline on a regression dataset

X_train_r, X_test_r, y_train_r, y_test_r, transformer_r = preprocess_dataset(
    task_id=361089,   # cpu_act
    suite_name='numerical_regression'
)

print(f"X_train shape: {X_train_r.shape}")
print(f"y_train dtype: {y_train_r.dtype}")
print(f"\ny_train stats after transformation:")
print(pd.Series(y_train_r).describe().round(4))

X_train shape: (8000, 8)
y_train dtype: float64

y_train stats after transformation:
count    8000.0000
mean        0.1381
std         1.4098
min        -5.1993
25%        -0.6737
50%         0.0013
75%         0.6750
max         5.1993
dtype: float64
